# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

This notebook completes all three exercise tasks using `global_energy_mix.csv`.

**Dataset:** `data/global_energy_mix.csv` from the course GitHub repository.

If you are using Google Colab, run the setup cell first. If the dataset is not already available, the cell will ask you to upload the CSV.


In [5]:
# Setup and dataset loading
import pandas as pd
import plotly.express as px
import numpy as np
from pathlib import Path

# Look for the course dataset in common local and Google Colab locations.
candidate_paths = [
    Path("../data/global_energy_mix.csv"),
    Path("data/global_energy_mix.csv"),
    Path("/content/data/global_energy_mix.csv"),
    Path("/content/global_energy_mix.csv"),
    Path("/content/drive/MyDrive/data/global_energy_mix.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

# In Google Colab, upload the CSV if it was not found automatically.
if data_path is None:
    try:
        from google.colab import files
        print("Please upload the course file: global_energy_mix.csv")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        uploaded_name = next(iter(uploaded))
        data_path = Path("/content") / uploaded_name
    except ImportError:
        raise FileNotFoundError(
            "Could not find global_energy_mix.csv. Put it in the project's data/ "
            "folder, or run this notebook in Google Colab and upload the CSV."
        )

df = pd.read_csv(data_path)

# Standardise common column-name variations while retaining the dataset's data.
column_lookup = {str(c).strip().lower().replace(" ", "_"): c for c in df.columns}
aliases = {
    "country": ["country"],
    "region": ["region"],
    "source": ["source", "energy_source", "fuel", "fuel_type"],
    "twh": ["twh", "energy_twh", "value_twh", "terawatt_hours"],
}
rename_map = {}
for standard_name, possible_names in aliases.items():
    for name in possible_names:
        if name in column_lookup:
            rename_map[column_lookup[name]] = standard_name.title() if standard_name != "twh" else "TWh"
            break
df = df.rename(columns=rename_map)

required_columns = {"Region", "Country", "Source", "TWh"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_columns)}. "
        f"Columns found: {list(df.columns)}"
    )

# Ensure TWh is numeric and remove rows without the fields needed for the charts.
df["TWh"] = pd.to_numeric(df["TWh"], errors="coerce")
df = df.dropna(subset=["Region", "Country", "Source", "TWh"]).copy()

# Source type mapping — reuse from lecture.
source_category = {
    "Coal": "Fossil",
    "Oil": "Fossil",
    "Natural Gas": "Fossil",
    "Nuclear": "Low-carbon",
    "Hydro": "Low-carbon",
    "Wind": "Renewable",
    "Solar": "Renewable",
    "Other Renewables": "Renewable",
}
df["Source"] = df["Source"].astype(str).str.strip()
df["Source_Type"] = df["Source"].map(source_category)

print(f"Loaded {len(df):,} rows from: {data_path}")
print("Columns:", list(df.columns))
display(df.head(10))

unmapped_sources = sorted(df.loc[df["Source_Type"].isna(), "Source"].unique())
if unmapped_sources:
    print("Note: source values not used in the three tasks:", unmapped_sources)


Loaded 103 rows from: /content/global_energy_mix.csv
Columns: ['Country', 'Region', 'Source', 'Share_pct', 'TWh', 'Source_Type']


,Country,Region,Source,Share_pct,TWh,Source_Type
0,United States,North America,Coal,10,1015.0,Fossil
1,United States,North America,Oil,35,3220.0,Fossil
2,United States,North America,Natural Gas,34,3083.0,Fossil
3,United States,North America,Nuclear,9,798.0,Low-carbon
4,United States,North America,Hydro,3,339.0,Low-carbon
5,United States,North America,Wind,4,413.0,Renewable
6,United States,North America,Solar,3,325.0,Renewable
7,United States,North America,Other Renewables,2,229.0,Renewable
8,China,Asia,Coal,58,6929.1,Fossil
9,China,Asia,Oil,18,1620.0,Fossil


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting.
- Use `path=['Region', 'Country', 'Source']` for the hierarchy.
- Colour encodes Coal, Oil and Natural Gas with a CVD-safe palette.
- Show TWh values in labels — no percentages.
- Grey out parent nodes (Region and Country level).
- Insight title names the region with the largest fossil-energy total.


In [2]:
# Task 1 — Fossil fuel treemap

df_fossil = df.loc[df["Source_Type"] == "Fossil"].copy()
if df_fossil.empty:
    raise ValueError(
        "No fossil-fuel rows found. Check that Source contains Coal, Oil and Natural Gas."
    )

fossil_mix = (
    df_fossil.groupby(["Region", "Country", "Source"], as_index=False)["TWh"]
    .sum()
)

fossil_by_country = (
    fossil_mix.groupby(["Region", "Country"], as_index=False)["TWh"]
    .sum()
)
top_fossil_country = fossil_by_country.loc[fossil_by_country["TWh"].idxmax()]
fossil_by_region = fossil_by_country.groupby("Region")["TWh"].sum()
top_fossil_region = fossil_by_region.idxmax()

# CVD-safe palette (blue, orange, bluish green).
fossil_palette = {
    "Coal": "#0072B2",
    "Oil": "#E69F00",
    "Natural Gas": "#009E73",
}

fig1 = px.treemap(
    fossil_mix,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map=fossil_palette,
    title=f"Fossil fuel use by country and source — {top_fossil_region} has the largest regional total",
)

# Colour source leaves by fuel; colour Region/Country parent nodes grey.
ids = list(fig1.data[0].ids)
labels = list(fig1.data[0].labels)
node_colours = []
for node_id, label in zip(ids, labels):
    level = str(node_id).count("/")
    if level >= 2:
        node_colours.append(fossil_palette.get(label, "#999999"))
    else:
        node_colours.append("#BDBDBD")

fig1.update_traces(
    marker=dict(colors=node_colours, line=dict(color="white", width=1)),
    texttemplate="%{label}<br>%{value:,.0f} TWh",
    hovertemplate="<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>",
)
fig1.update_layout(margin=dict(t=75, l=10, r=10, b=10))
fig1.show()

print(
    f"Largest fossil-energy country: {top_fossil_country['Country']} "
    f"({top_fossil_country['Region']}), {top_fossil_country['TWh']:,.1f} TWh."
)


Largest fossil-energy country: Saudi Arabia (Middle East), 9,671.0 TWh.


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build both a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country.
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `Low-carbon`.
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour.
- Both charts show TWh values, not percentages.
- Insight title on the bar chart names the leading country.


In [3]:
# Task 2 — Low-carbon treemap and horizontal bar chart

df_low_carbon = df.loc[df["Source_Type"] == "Low-carbon"].copy()
low_carbon_by_country = (
    df_low_carbon.groupby("Country", as_index=False)["TWh"]
    .sum()
    .sort_values("TWh", ascending=False)
)

if low_carbon_by_country.empty:
    raise ValueError("No low-carbon rows found. Check that Source contains Nuclear and Hydro.")

low_carbon_by_country["All"] = "Low-carbon"
leading_country = low_carbon_by_country.iloc[0]

# Treemap: rectangle area represents total low-carbon energy in TWh.
fig2a = px.treemap(
    low_carbon_by_country,
    path=["All", "Country"],
    values="TWh",
    color="TWh",
    color_continuous_scale="Teal",
    title="Low-carbon energy by country (Nuclear + Hydro)",
)
fig2a.update_traces(
    texttemplate="%{label}<br>%{value:,.0f} TWh",
    hovertemplate="<b>%{label}</b><br>%{value:,.1f} TWh<extra></extra>",
)
fig2a.update_layout(margin=dict(t=65, l=10, r=10, b=10))
fig2a.show()

# Horizontal bars sorted from smallest at the bottom to largest at the top.
fig2b = px.bar(
    low_carbon_by_country,
    x="TWh",
    y="Country",
    orientation="h",
    color_discrete_sequence=["#0072B2"],
    title=f"Low-carbon energy by country — {leading_country['Country']} leads",
    labels={"TWh": "Low-carbon energy (TWh)", "Country": "Country"},
)
fig2b.update_traces(
    texttemplate="%{x:,.0f} TWh",
    textposition="outside",
    cliponaxis=False,
    hovertemplate="<b>%{y}</b><br>%{x:,.1f} TWh<extra></extra>",
)
fig2b.update_layout(
    yaxis={"categoryorder": "total ascending"},
    margin=dict(t=75, l=10, r=55, b=10),
)
fig2b.show()

print(
    f"Leading country: {leading_country['Country']} "
    f"({leading_country['TWh']:,.1f} TWh)."
)


Leading country: France (7,593.0 TWh).


### Task 2 — Reflection

The **horizontal bar chart** communicates the ranking more clearly because all countries share the same baseline, so their TWh values can be compared directly by bar length. Sorting the bars also makes the highest and lowest countries immediately visible. The treemap is useful for showing each country's relative contribution to the whole, but comparing similarly sized rectangles is less precise when the main question is the ranking.


## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each Region, how its fossil energy splits between Coal, Oil and Natural Gas. This is the same data as the Task 1 treemap, visualised a second way.

**Requirements:**
- Reuse `df_fossil`; aggregate TWh by `Region` × `Source` and convert to shares of each region's fossil total.
- Horizontal bars, regions sorted by coal share (highest at the top).
- Coal is the first segment and the only coloured segment; Oil and Natural Gas are grey.
- Percentage labels on coal segments only.
- Insight title names the most coal-dependent region and its coal share.


In [4]:
# Task 3 — Regional fossil-fuel shares (100% stacked bar)

regional_source = (
    df_fossil.groupby(["Region", "Source"], as_index=False)["TWh"]
    .sum()
)

regional_source["Region_Total"] = (
    regional_source.groupby("Region")["TWh"].transform("sum")
)
regional_source["Share"] = (
    regional_source["TWh"] / regional_source["Region_Total"] * 100
)

coal_share = (
    regional_source.loc[regional_source["Source"] == "Coal", ["Region", "Share"]]
    .set_index("Region")["Share"]
)

if coal_share.empty:
    raise ValueError("No Coal rows found in the fossil-fuel dataset.")

# Include all regions; if a region has no Coal row, its coal share is zero.
all_regions = regional_source["Region"].drop_duplicates().tolist()
coal_share = coal_share.reindex(all_regions, fill_value=0).sort_values(ascending=False)
region_order = coal_share.index.tolist()

top_coal_region = coal_share.idxmax()
top_coal_percentage = coal_share.max()

source_order = ["Coal", "Oil", "Natural Gas"]
regional_source["Source"] = pd.Categorical(
    regional_source["Source"],
    categories=source_order,
    ordered=True,
)

stacked_palette = {
    "Coal": fossil_palette["Coal"],
    "Oil": "#A6A6A6",
    "Natural Gas": "#D9D9D9",
}

fig3 = px.bar(
    regional_source,
    x="Share",
    y="Region",
    color="Source",
    orientation="h",
    barmode="stack",
    category_orders={
        "Source": source_order,
        "Region": region_order[::-1],
    },
    color_discrete_map=stacked_palette,
    title=f"Coal is the largest fossil share in {top_coal_region} ({top_coal_percentage:.1f}%)",
    labels={"Share": "Share of regional fossil energy (%)", "Region": "Region"},
)

# Percentage labels are displayed only on Coal segments.
fig3.update_traces(
    selector=dict(name="Coal"),
    texttemplate="%{x:.1f}%",
    textposition="inside",
    hovertemplate="<b>%{y}</b><br>Coal: %{x:.1f}%<extra></extra>",
)
fig3.update_traces(
    selector=dict(name="Oil"),
    hovertemplate="<b>%{y}</b><br>Oil: %{x:.1f}%<extra></extra>",
)
fig3.update_traces(
    selector=dict(name="Natural Gas"),
    hovertemplate="<b>%{y}</b><br>Natural Gas: %{x:.1f}%<extra></extra>",
)

fig3.update_layout(
    xaxis=dict(range=[0, 100], ticksuffix="%", title="Share of regional fossil energy"),
    yaxis=dict(title="Region"),
    legend_title_text="Fossil source",
    margin=dict(t=80, l=10, r=10, b=10),
)
fig3.show()

print(
    f"Most coal-dependent region: {top_coal_region} "
    f"({top_coal_percentage:.1f}% of its fossil energy)."
)


Most coal-dependent region: Africa (78.0% of its fossil energy).


### Task 3 — Reflection

The **treemap** answers: *Which countries and regions account for the largest amounts of fossil-fuel use, and how are those amounts divided among Coal, Oil and Natural Gas?* It emphasizes absolute TWh and hierarchical contribution.

The **100% stacked bar chart** answers: *Which region has the highest proportion of coal in its fossil-energy mix?* It compares the composition of each region's fossil energy, regardless of the region's total size.
